# 📈 Prévoir les bonds des joueurs d'une saison à la suivante

**Question :** à partir de tout ce qu'on sait d'un joueur à la fin d'une saison — ses stats, son profil, mais aussi
son équipe de la saison suivante, sa ville, son coach — peut-on prévoir **qui va faire un bond analytique** ?

- **Mesure de niveau :** le **BPM** (Box Plus/Minus, Basketball-Reference) ; 0 = joueur moyen, +5 = All-Star.
- **Cible :** `d_bpm` = BPM(saison suivante) − BPM(cette saison), pour les joueurs ayant joué **≥ 500 minutes les deux saisons**.
- **Un « bond » :** gain d'au moins **+2 de BPM** (≈ 14 % des paires).
- **Données :** 12 saisons (2014-15 → 2025-26) → 11 paires de saisons consécutives, ~3 150 paires joueur ;
  **~300 variables** en 9 familles (profil, box score, stats avancées, scoring, hustle, tracking, tirs, tendance, contexte).
- **Contexte** (connu avant le début de la saison suivante) : équipe de la saison suivante, **ville** (taille du marché,
  altitude), changement d'équipe, niveau de l'équipe, minutes libérées par les départs, usage des coéquipiers restants,
  **coach principal** (changement, ancienneté, taille du staff, entraîneurs « développement », effet passé du coach
  et de la franchise sur la progression des joueurs — calculé uniquement sur les saisons antérieures).
- Code : `progression.py` (`python main.py --step progression`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import progression as P

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
GRAY, INK, MUTED = "#c9c8c3", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRAY, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": "#eeeeec", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titleweight": "bold", "axes.titlecolor": INK, "axes.titlelocation": "left",
})

pairs, fam = P.build_pairs()                 # ~30 s
feats = P.all_features(fam)
lab = pairs.dropna(subset=["d_bpm"])
print(f"{len(lab)} paires étiquetées, {lab['breakout'].mean():.0%} de bonds (ΔBPM ≥ {P.BREAKOUT}) — {len(feats)} variables")
pd.Series({f: len(fam[f]) for f in P.FAMILIES}, name="variables").to_frame().T

## 1. Quelles stats annoncent un bond ?

Un piège domine tout : la **régression vers la moyenne**. Un joueur qui vient d'avoir une saison très basse remonte
souvent, une saison exceptionnelle est rarement répétée. Pour savoir ce qu'une stat apporte *vraiment*, on mesure
sa corrélation avec la part du bond **que la référence (BPM, âge, minutes, BPM de la saison d'avant) n'explique pas**.

In [ ]:
univ = P.univariate_signal(pairs, feats)
fam_of = {c: f for f in P.FAMILIES for c in fam[f]}
univ["famille"] = univ["variable"].map(fam_of)
top = univ.head(25)
display(top.round(3))

colors = {"tirs": ORANGE, "tracking": BLUE, "box": AQUA, "avance": "#8a5cd6", "scoring": "#d6a35c",
          "tendance": MUTED, "contexte": "#d65c8a", "joueur": "#5cb8d6", "hustle": "#7a9a3a"}
t = top[::-1]
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(t["variable"], t["signal en plus de la référence"], color=[colors[f] for f in t["famille"]], height=0.7)
ax.axvline(0, color=GRAY, lw=1)
ax.set_xlabel("Corrélation (Spearman) avec le bond non expliqué par la référence")
ax.set_title("Les 25 stats qui en disent le plus sur le bond à venir")
for f, c in colors.items():
    if f in set(t["famille"]):
        ax.barh([], [], color=c, label=f)
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**Lecture (signe négatif = annonce une baisse, positif = annonce une hausse) :**

- **La réussite à 3 points est le premier signal… à l'envers.** Un fort % à 3 pts (surtout en catch & shoot
  et dans les corners) annonce une **baisse** l'année suivante : le tir à 3 est très bruité d'une saison à l'autre,
  le BPM d'un tireur adroit retombe.
- **Le jeu intérieur et la création tiennent :** rebonds défensifs, jeu au poste (touches, passes, paniers),
  fautes provoquées, part des points à 2 pts annoncent une **hausse** relative. Les **ballons perdus** aussi :
  beaucoup de pertes = un joueur à qui on confie la création, signe d'un rôle qui grandit.
- **La tendance récente se corrige :** un gros bond de BPM la saison passée (`tend_bpm_delta`) est en partie rendu ensuite.

In [ ]:
ctx = univ[univ["famille"] == "contexte"].reset_index(drop=True)
ctx.round(3)

**Ville, équipe, coach :**

- **Changer d'équipe** (−0,09) et **changer de coach** (−0,07) annoncent une progression plus faible ; l'**ancienneté
  du coach** (+0,07) une progression plus forte. La stabilité aide.
- La **ville** compte peu : marché un peu défavorable dans les grandes villes (−0,03), léger plus en altitude
  (Denver, Utah, +0,03) — effets faibles, à la limite du bruit.
- L'**effet passé du coach** et de la **franchise** (les joueurs ont-ils progressé chez eux auparavant ?) n'apporte
  presque rien (+0,02 / +0,05) : il n'y a pas, dans ces données, de « coach qui fait progresser » repérable à l'avance.
- Le **niveau de l'équipe** n'apporte rien une fois la référence connue.

In [ ]:
fi = P.family_importance(pairs, fam)        # ~1-2 min : on mélange chaque famille et on mesure la perte
display(fi.round(4))
f2 = fi[::-1]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(f2["famille"], f2["baisse de Spearman"], color=BLUE, height=0.6)
ax.set_xlabel("Baisse du Spearman (tests 2019-20 → 2025-26) quand la famille est mélangée")
ax.set_title("Importance de chaque famille de stats dans le modèle")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

Dans le modèle complet, les **stats avancées** (qui contiennent le BPM lui-même, donc la régression vers la moyenne)
pèsent de loin le plus, puis la **tendance**, le **profil** (âge surtout), le **tracking** et le **contexte**.
Hustle, scoring et zones de tir n'ajoutent rien *dans le modèle* : leur information est déjà portée par le
tracking et le box score (et les zones de tir n'existent qu'à partir de 2021-22).

## 2. Les modèles prévoient-ils les bonds ? (validation dans le temps)

Chaque saison de 2019-20 à 2025-26 est prédite par un modèle entraîné **uniquement sur les saisons précédentes**,
comme en conditions réelles.

In [ ]:
res = P.evaluate_rolling(pairs, feats)
display(res["global"].round(3))

per = res["par_saison"].pivot(index="saison testée", columns="modèle", values="Spearman")
fig, ax = plt.subplots(figsize=(8, 3.8))
for m, c in [("Référence : niveau, âge, minutes (linéaire)", GRAY), (P.ENSEMBLE, BLUE)]:
    ax.plot(per.index, per[m], marker="o", color=c, label=m, lw=2)
ax.set_ylabel("Spearman (bond prédit vs réel)")
ax.set_title("Chaque saison prédite à partir des précédentes")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

- La **référence seule** (BPM, âge, minutes) atteint déjà un Spearman ≈ 0,45 : l'essentiel de ce qui est prévisible,
  c'est la régression vers la moyenne et l'âge (les jeunes progressent, les plus de 30 ans baissent).
- **Toutes les stats** ajoutent un gain modeste mais régulier : Spearman ≈ 0,49 pour la moyenne linéaire + boosting,
  AUC « bond » 0,77 contre 0,73.
- **Concrètement :** parmi les 25 joueurs que le modèle classe en tête chaque saison, **≈ 45 % font un vrai bond**
  (≥ +2 BPM), contre 15 % en moyenne et ≈ 41 % pour la référence.

## 3. Essai sur la saison 2026-27

Modèle entraîné sur toutes les paires jusqu'à 2024-25 → 2025-26, appliqué aux stats 2025-26 avec le **contexte
2026-27** (effectifs et coachs de début de saison). On compare ensuite au **BPM des ~18 premiers matchs** de 2026-27 :
c'est un vrai test hors échantillon, mais très bruité (peu de minutes).

In [ ]:
import sqlite3
nxt = P.predict_next(pairs, feats)
ref = P.predict_next(pairs, feats, "Référence : niveau, âge, minutes (linéaire)")
nxt["d_ref"] = ref["d_bpm_predit"].values
nxt["apport_des_stats"] = nxt["d_bpm_predit"] - nxt["d_ref"]
conn = sqlite3.connect("file:nba_draft_analytics.db?mode=ro", uri=True)
names = pd.read_sql_query("SELECT player_id, full_name FROM players", conn)
obs = pd.read_sql_query("SELECT player_id, MAX(mp) AS mp_2627, bpm AS bpm_2627 FROM bbref_advanced "
                        "WHERE season = '2026-27' AND player_id IS NOT NULL GROUP BY player_id", conn)
conn.close()
nxt = nxt.merge(names, on="player_id", how="left").merge(obs, on="player_id", how="left")
cols = ["full_name", "age", "ctx_equipe", "ctx_change_equipe", "bpm", "d_bpm_predit", "bpm_predit", "mp_2627", "bpm_2627"]

print("Plus gros bonds prédits parmi les joueurs de rotation (≥ 1 500 min en 2025-26) :")
display(nxt[nxt["mp"] >= 1500].nlargest(12, "d_bpm_predit")[cols].round(1))
print("Ce que les stats ajoutent à la référence — révisés à la hausse :")
display(nxt.nlargest(10, "apport_des_stats")[cols + ["box_fg3_pct", "apport_des_stats"]].round(2))
print("… et à la baisse :")
display(nxt.nsmallest(10, "apport_des_stats")[cols + ["box_fg3_pct", "apport_des_stats"]].round(2))

- Les plus gros bonds prédits chez les joueurs de rotation sont des **jeunes de 19-22 ans** après une première saison
  difficile (Will Riley, Jeremiah Fears, Ace Bailey, Cooper Flagg…) : c'est l'âge + la régression vers la moyenne.
- Les **révisions à la baisse** par rapport à la référence touchent surtout des **tireurs très adroits en 2025-26**
  (Luke Kennard, Isaiah Joe, Tim Hardaway Jr.…) et des joueurs qui **changent d'équipe** : exactement les signaux
  trouvés en section 1.

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
t = nxt[nxt["mp_2627"] >= 150].copy()
t["d_obs"] = t["bpm_2627"] - t["bpm"]
t["bond_obs"] = (t["d_obs"] >= P.BREAKOUT).astype(int)
rows = []
for nom, col in [("Référence : BPM, âge, minutes", "d_ref"), ("Toutes les stats (moyenne lin. + boosting)", "d_bpm_predit")]:
    rows.append({"modèle": nom, "Spearman": spearmanr(t["d_obs"], t[col]).statistic,
                 "AUC bond": roc_auc_score(t["bond_obs"], t[col]),
                 "bonds dans le top 25": t.nlargest(25, col)["bond_obs"].mean()})
print(f"{len(t)} joueurs avec ≥ 150 minutes en 2026-27 (au 23 nov.) — taux de bonds observé : {t['bond_obs'].mean():.0%}")
pd.DataFrame(rows).set_index("modèle").round(3)

**Sur le début de saison 2026-27, le modèle complet bat la référence** (Spearman ≈ 0,31 contre ≈ 0,26). Les scores sont
plus bas qu'en validation parce que le BPM sur ~18 matchs est très bruité ; à refaire en fin de saison
(`python main.py --step bbref --seasons 2026-27` puis relancer ce notebook).

## 4. Un indice de réussite moins sensible à la réussite au tir

Le ΔBPM est très lié à la variation du 3 points (le signal n°1 ci-dessus). On construit donc un **indice de réussite**
(`indice.py`), **par minute** (centré sur la minute NBA moyenne de la saison), en trois piliers de même poids :

| Pilier | Composantes (z-scores de la saison, pondérés par les minutes) |
|---|---|
| **Efficacité offensive** | points ajoutés par l'efficacité (TS% vs ligue × volume, **3 pts ramené vers la moyenne** : +250 tirs fictifs à la réussite de la ligue), offensive rating sur le terrain, pertes de balle de l'équipe (inversé) |
| **Efficacité défensive** | defensive rating sur le terrain (inversé), interceptions, contres, % rebonds défensifs, ballons déviés |
| **Production brute** | points, rebonds, passes, pertes de balle (inversé) par 36 min |

L'indice est remis sur l'échelle du BPM (0 = minute moyenne, même dispersion) pour comparer les deux.
`indice_volume` = indice × minutes relatives, si l'on veut aussi récompenser le temps de jeu.

In [ ]:
import sqlite3
pairs_ix, _ = P.build_pairs(target="indice")       # mêmes paires, effets coach / franchise ré-estimés sur l'indice
lab_ix = pairs_ix.dropna(subset=["d_indice"])
conn = sqlite3.connect("file:nba_draft_analytics.db?mode=ro", uri=True)
names = pd.read_sql_query("SELECT player_id, full_name FROM players", conn)
tnet = pd.read_sql_query("SELECT team_id, season, net_rating FROM team_stats_advanced", conn)
conn.close()

panel, _ = P.load_panel()
nxt_panel = panel[["player_id", "start", "box_fg3_pct", "box_fg3a36"]].assign(start=lambda d: d["start"] - 1)
l2 = lab_ix.merge(nxt_panel, on=["player_id", "start"], suffixes=("", "_next"))
l2["d_3pts"] = l2["box_fg3_pct_next"] - l2["box_fg3_pct"]
tireurs = l2["box_fg3a36"] >= 3
team = (panel[(panel["mp_nba"] > 0) & (panel["season"] <= "2025-26")]
        .groupby(["team_id", "season"])
        .apply(lambda g: pd.Series({k: np.average(g[k].fillna(g[k].median()), weights=g["mp_nba"])
                                    for k in ["indice", "bpm"]}))
        .reset_index().merge(tnet))
display(pd.DataFrame({
    "BPM": [lab_ix[["bpm", "bpm_next"]].corr().iloc[0, 1], l2[tireurs][["d_3pts", "d_bpm"]].corr().iloc[0, 1],
            team[["bpm", "net_rating"]].corr().iloc[0, 1]],
    "Indice de réussite": [lab_ix[["indice", "indice_next"]].corr().iloc[0, 1],
                           l2[tireurs][["d_3pts", "d_indice"]].corr().iloc[0, 1],
                           team[["indice", "net_rating"]].corr().iloc[0, 1]]},
    index=["stabilité d'une saison à l'autre (corr. t / t+1)",
           "lien de la variation avec la variation du 3 pts (tireurs ≥ 3 tirs / 36)",
           "corr. de la moyenne d'équipe (pondérée minutes) avec le net rating"]).round(2))
p25 = panel[(panel["season"] == "2025-26") & (panel["mp_nba"] >= 1500)].merge(names, on="player_id")
print("Meilleurs indices 2025-26 (≥ 1 500 min) :")
display(p25.nlargest(15, "indice")[["full_name", "indice", "bpm", "ix_efficacite_off", "ix_efficacite_def",
                                    "ix_production"]].round(1).reset_index(drop=True))

- **La variation de l'indice dépend presque deux fois moins de la variation au 3 pts** (corr. ≈ 0,29 contre ≈ 0,50 pour
  le ΔBPM) : c'est l'effet du 3 pts ramené vers la moyenne.
- Il reste **très lié à la réussite des équipes** (≈ 0,94 avec le net rating ; le BPM est construit pour coller au
  net rating, d'où son ≈ 0,98) mais il est un peu **moins stable** d'une saison à l'autre (≈ 0,70 contre ≈ 0,76).
  Les ratings sur le terrain sont bruités et dépendent des coéquipiers.
- **Biais :** il favorise les **pivots** (rebonds, contres, % rebonds, efficacité près du cercle) et sous-note les
  créateurs à gros volume (Dončić ≈ 5 contre 9 au BPM), car le volume de création et l'usage n'y figurent pas et les
  pertes de balle sont pénalisées deux fois.

In [ ]:
res_ix = P.evaluate_rolling(pairs_ix, feats)
display(res_ix["global"].round(3))
univ_ix = P.univariate_signal(pairs_ix, feats)
univ_ix["famille"] = univ_ix["variable"].map(fam_of)
print("Signaux les plus forts pour le bond d'indice (en plus de la référence : indice, âge, minutes) :")
display(univ_ix.head(15).round(3))
print("Contexte :")
display(univ_ix[univ_ix["famille"] == "contexte"].round(3).reset_index(drop=True))

**Validation dans le temps (2019-20 → 2025-26), cible = Δ indice (bond = +2 points d'indice, ≈ 17 % des cas) :**
le modèle complet atteint un **Spearman ≈ 0,51** contre ≈ 0,45 pour la référence, et **56 % de vrais bonds parmi
ses 25 premiers** contre 50 % : l'écart avec la référence est un peu plus grand qu'avec le BPM.

**Signaux :** le **jeu intérieur** passe en tête (touches au poste, dans la raquette et au coude, rebonds, pertes de
balle de l'équipe). Le **corner 3** et la distance de tir moyenne annoncent toujours une baisse. Côté contexte, les
**minutes libérées** (−0,12), le **changement d'équipe** (−0,10) et le **changement de coach** (−0,08) annoncent une
baisse, l'**ancienneté du coach** (+0,08) une hausse. La ville et l'« effet coach » passé restent ≈ 0.

In [ ]:
from scipy.stats import spearmanr
nx = P.predict_next(pairs_ix, feats)
nx["d_ref"] = P.predict_next(pairs_ix, feats, "Référence : niveau, âge, minutes (linéaire)")["d_indice_predit"].values
cur = panel[panel["season"] == "2026-27"][["player_id", "indice", "mp_nba"]].rename(
    columns={"indice": "indice_2627", "mp_nba": "mp_2627"})
nx = nx.merge(names, on="player_id", how="left").merge(cur, on="player_id", how="left")
cols = ["full_name", "age", "ctx_equipe", "bpm", "indice", "ix_efficacite_off", "ix_efficacite_def",
        "ix_production", "d_indice_predit", "indice_predit", "indice_2627"]
print("Plus gros bonds d'indice prédits (≥ 1 500 min en 2025-26) :")
display(nx[nx["mp"] >= 1500].nlargest(12, "d_indice_predit")[cols].round(1))
print("Meilleurs indices prédits pour 2026-27 :")
display(nx[nx["mp"] >= 1500].nlargest(12, "indice_predit")[cols].round(1))
t = nx[nx["mp_2627"] >= 150].copy()
t["d_obs"] = t["indice_2627"] - t["indice"]
print(f"Début de saison 2026-27 ({len(t)} joueurs ≥ 150 min) — Spearman avec le bond observé : "
      f"modèle {spearmanr(t['d_obs'], t['d_indice_predit']).statistic:.2f}, "
      f"référence {spearmanr(t['d_obs'], t['d_ref']).statistic:.2f}")

- Les plus gros bonds d'indice prédits sont de nouveau des **jeunes** (Bub Carrington, Will Riley, Jeremiah Fears,
  Ace Bailey, Banchero) et des joueurs d'**Indiana** (Jarace Walker, Nembhard, Siakam, Huff), qui récupèrent des
  minutes et des possessions (Haliburton blessé en 2025-26).
- **Sur le début de 2026-27**, le modèle et la référence font jeu égal sur l'indice (≈ 0,30 contre ≈ 0,29). L'apport des
  stats ne se voit pas encore sur ~18 matchs.

## 5. Limites

- **Sélection :** seuls les joueurs ayant joué ≥ 500 minutes les deux saisons comptent ; un joueur qui s'effondre
  au point de ne plus jouer sort de l'échantillon.
- **Le BPM** est une mesure de box score : il récompense rebonds et création, pénalise les spécialistes défensifs.
- **Contexte imparfait :** l'équipe de la saison suivante est celle de l'effectif (en fin de saison pour l'historique,
  en début de saison pour 2026-27) ; les coachs 2025-26 de Chicago, Dallas et Orlando manquent dans l'API.
- **Manquent :** blessures, contrats (année de contrat), changements de rôle annoncés, travail d'été.
- **L'indice de réussite** donne le même poids à ses trois piliers, un choix arbitraire. On pourrait apprendre les poids
  (par exemple pour coller au net rating sur le terrain) et centrer aussi par poste pour corriger le biais pivots.